# Basic GAN for Synthetic Image Generation

In this project, we build a Generative Adversarial Network (GAN)
to generate synthetic handwritten digit images.

The GAN consists of two neural networks:

1. Generator - creates fake images from random noise.
2. Discriminator - distinguishes between real and fake images.

The two networks are trained together so that the Generator
gradually learns to create images that look like real MNIST digits.

In [10]:
import tensorflow as tf 
from tensorflow import keras 
from tensorflow.keras import layers
import numpy as np


In [11]:
#Load Dataset
(X_train,_),(_,_) = tf.keras.datasets.mnist.load_data()
X_train = X_train.astype('float32') / 255.0 
X_train = np.reshape(X_train, (-1,28,28,1))

In [12]:
#Generator 
latent_dim = 100 

generator = keras.Sequential([
    layers.Input(shape=(latent_dim,)),
    layers.Dense(128, activation='relu'),
    layers.Dense(28*28, activation='sigmoid'),
    layers.Reshape((28,28,1))
])
generator.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_4 (Dense)                 │ (None, 128)            │        12,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 784)            │       101,136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape_1 (Reshape)             │ (None, 28, 28, 1)      │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 114,064 (445.56 KB)

 Trainable params: 114,064 (445.56 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
#Discriminator
discriminator = keras.Sequential([
    layers.Input(shape=(28,28,1)),
    layers.Flatten(),
    layers.Dense(128,activation='relu'),
    layers.Dense(1, activation='sigmoid')
])
discriminator.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_1 (Flatten)             │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 100,609 (393.00 KB)

 Trainable params: 100,609 (393.00 KB)

 Non-trainable params: 0 (0.00 B)

In [14]:
discriminator.compile(
    optimizer = keras.optimizers.Adam(learning_rate=0.0002),
    loss = 'binary_crossentropy',
    metrics=['accuracy']
)

In [15]:
#Combine Generator + Discriminator

discriminator.trainable = False 

gan_input = keras.Input(
    shape=(latent_dim,)
)

fake_image = generator(gan_input)

gan_output = discriminator(fake_image)

gan = keras.Model(gan_input, gan_output)

gan.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0002),
            loss='binary_crossentropy',
            metrics=['accuracy']
            )
gan.summary()


Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)      │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_2 (Sequential)       │ (None, 28, 28, 1)      │       114,064 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_3 (Sequential)       │ (None, 1)              │       100,609 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 214,673 (838.57 KB)

 Trainable params: 114,064 (445.56 KB)

 Non-trainable params: 100,609 (393.00 KB)

In [16]:
#Training 
epochs = 5000
batch_size = 128 

for epoch in range(epochs):
    #Train Discriminator
    idx = np.random.randint(0, X_train.shape[0],batch_size)
    real_images = X_train[idx]
    noise = np.random.normal(0,1, (batch_size, latent_dim))
    fake_images = generator.predict(noise, verbose=0)

    d_loss_real = discriminator.train_on_batch(real_images, np.ones((batch_size,1)))
    d_loss_fake = discriminator.train_on_batch(fake_images, np.zeros((batch_size,1)))

    #Train Generator 
    noise = np.random.normal(0,1, (batch_size, latent_dim))

    g_loss = gan.train_on_batch(noise, np.ones((batch_size,1)))

    if epoch % 1000 == 0:
        print(
            f"Epoch:{epoch} |"
            f"D Loss: {d_loss_real[0]:.4f} |"
            f"G Loss: {g_loss[0]:.4f}"
        )

Epoch:0 |D Loss: 0.4581 |G Loss: 0.5164
Epoch:1000 |D Loss: 3.2127 |G Loss: 0.0141
Epoch:2000 |D Loss: 3.7372 |G Loss: 0.0073
Epoch:3000 |D Loss: 4.0564 |G Loss: 0.0049
Epoch:4000 |D Loss: 4.2944 |G Loss: 0.0037
